<a href="https://github.com/n3iKos/segsmaker-fast">
  <img alt="GitHub repo" src="https://img.shields.io/badge/GitHub-segsmaker--fast-6e5494?style=for-the-badge&logo=github&logoColor=white"/>
</a><br>

* get your Civitai API key from [here](https://civitai.com/user/account)
* get your Hugging Face read token from [here](https://huggingface.co/settings/tokens)


In [ ]:
# @title <b><font color='orange'>WebUI Installer</font></b> { display-mode: "form" }
# @markdown <details open><summary><b>Step 1 - Choose WebUI</b></summary>
Webui = 'Forge-Neo' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
# @markdown </details>
# @markdown <details open><summary><b>Step 2 - API Keys</b></summary>
# @markdown [Get Civitai key](https://civitai.com/user/account) &nbsp; | &nbsp; [Get Hugging Face token](https://huggingface.co/settings/tokens)
Civitai__Key = '' # @param { type: "string", placeholder: "Your Civitai API Key (required)" }
HF_Read_Token = '' # @param { type: "string", placeholder: "Your Hugging Face READ Token (optional)" }
# @markdown </details>
# @markdown <details open><summary><b>Step 3 - Google Drive</b></summary>
Mount_GDrive = 'No' # @param ["No", "Yes"]
# @markdown </details>

from pathlib import Path
import os, json, shlex, subprocess, sys, re, tempfile, shutil, threading
from urllib.parse import urlparse

FAST_REPO = 'https://github.com/n3iKos/segsmaker-fast'
mount = Mount_GDrive

if mount == 'Yes':
    from google.colab import drive
    drive.mount('/content/drive')

!curl -sLo /content/setup.py {FAST_REPO}/raw/main/script/KC/setup.py
%run /content/setup.py --webui="$Webui" --civitai_key="$Civitai__Key" --hf_read_token="$HF_Read_Token"

if mount == 'Yes':
    from pathlib import Path

    d = Path('/content/drive/MyDrive/Segsmaker')

    for n, p in {'checkpoint': CKPT, 'lora': LORA, 'vae': VAE, 'embeddings': Embeddings}.items():
        if p is None:
            continue
        f = d / n
        f.mkdir(parents=True, exist_ok=True)
        s = p / f'drive-{n}'
        if not s.exists():
            s.symlink_to(f, target_is_directory=True)

    !rm -rf $WebUI_Output
    o = d / {'ComfyUI': 'comfyui-output', 'SwarmUI': 'swarmui-output'}.get(Webui, 'output')
    o.mkdir(parents=True, exist_ok=True)
    if not WebUI_Output.exists():
        WebUI_Output.symlink_to(o, target_is_directory=True)

    if Webui not in {'ComfyUI', 'SwarmUI'}:
        wc = WebUI / 'cache'
        !rm -rf $wc
        c = d / 'cache'
        c.mkdir(parents=True, exist_ok=True)
        if not wc.exists():
            wc.symlink_to(c, target_is_directory=True)

# Segsmaker Fast helper functions. They are intentionally kept in this cell so the following form cells stay simple.
def _sm_strip(values):
    return [str(v).strip() for v in values if str(v).strip()]

def _sm_import_downloader():
    try:
        import nenen88
        return nenen88
    except Exception:
        startup = Path('/root/.ipython/profile_default/startup/nenen88.py')
        if startup.exists():
            get_ipython().run_line_magic('run', str(startup))
            import nenen88
            return nenen88
        raise RuntimeError('Downloader belum tersedia. Jalankan ulang Cell 1 sampai instalasi selesai.')

def _sm_path(var_name, fallback=None):
    value = globals().get(var_name, fallback)
    if value is None:
        raise RuntimeError(f'Folder {var_name} belum tersedia untuk WebUI ini. Jalankan Cell 1, atau pilih WebUI yang mendukung folder tersebut.')
    path = Path(value)
    path.mkdir(parents=True, exist_ok=True)
    return path

def _sm_find_output_name(url, custom_name=None):
    if custom_name:
        return custom_name
    name = Path(urlparse(url).path).name
    return name or None

def _sm_resolve_record(line, target_dir):
    nenen88 = _sm_import_downloader()
    import requests

    parts = shlex.split(str(line).strip())
    if not parts:
        return None

    raw_url = parts[0].replace('\\', '')
    custom_name = parts[1] if len(parts) >= 2 and '/' not in parts[1] and not parts[1].startswith('~') else None

    # gdown has its own flow; keep Google Drive on the original downloader.
    if 'drive.google.com' in raw_url:
        return {'fallback': True, 'line': line, 'target': Path(target_dir)}

    resolved_url, civitai_json, version_id, filename = nenen88.get_url(raw_url, custom_name)
    if not resolved_url:
        return None

    civitai = nenen88.get_civdom(resolved_url)
    user_agent = nenen88.civitai_headers()['User-Agent'] if civitai else 'Mozilla/5.0'
    headers = [f'User-Agent: {user_agent}']

    if getattr(nenen88, 'TOBRUT', '') and 'huggingface.co' in resolved_url:
        headers.append(f"Authorization: Bearer {nenen88.TOBRUT}")

    if getattr(nenen88, 'TOKET', '') and civitai and f'{civitai}/api/download/models/' in resolved_url:
        headers.append(f"Authorization: Bearer {nenen88.TOKET}")
        try:
            response = requests.get(
                resolved_url,
                headers={'User-Agent': user_agent, 'Authorization': f'Bearer {nenen88.TOKET}'},
                allow_redirects=True,
                stream=True,
                timeout=30,
            )
            if response.url and response.url != resolved_url:
                resolved_url = response.url
                headers = [h for h in headers if not h.startswith('Authorization:')]
            response.close()
        except Exception as exc:
            print(f'  Preflight Civitai gagal, lanjut memakai header Authorization: {exc}')

    filename = filename or _sm_find_output_name(resolved_url, custom_name)
    return {
        'fallback': False,
        'url': resolved_url,
        'target': Path(target_dir),
        'filename': filename,
        'headers': headers,
        'civitai_json': civitai_json,
        'version_id': version_id,
        'input': line,
    }

def _sm_format_aria(line):
    text = line.rstrip('\n')
    text = re.sub(r'\[(#[^\]]+)\]', r'【\1】', text)
    return text + ('\n' if line.endswith('\n') else '')

def _sm_run_aria(records, max_workers=3):
    if not records:
        return
    if shutil.which('aria2c') is None:
        subprocess.run(shlex.split('sudo apt-get -qq -y install aria2'), check=False)

    with tempfile.NamedTemporaryFile('w', delete=False, encoding='utf-8') as handle:
        input_file = handle.name
        for rec in records:
            rec['target'].mkdir(parents=True, exist_ok=True)
            handle.write(rec['url'] + '\n')
            handle.write(f"  dir={rec['target']}\n")
            if rec.get('filename'):
                handle.write(f"  out={rec['filename']}\n")
            handle.write('  allow-overwrite=true\n')
            for header in rec.get('headers', []):
                handle.write(f'  header={header}\n')

    max_workers = max(1, int(max_workers))
    cmd = [
        'aria2c',
        f'--input-file={input_file}',
        f'-j{max_workers}',
        '-x16', '-s16', '-k1M', '-c',
        '--auto-file-renaming=false',
        '--allow-overwrite=true',
        '--console-log-level=notice',
        '--summary-interval=1',
        '--download-result=hide',
        '--stderr=true',
    ]

    print(f'  Parallel download: {len(records)} file(s), Max_Workers={max_workers}')
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc.stdout:
            sys.stdout.write(_sm_format_aria(line))
            sys.stdout.flush()
    finally:
        proc.wait()
        Path(input_file).unlink(missing_ok=True)

    if proc.returncode != 0:
        raise RuntimeError(f'aria2c berhenti dengan kode {proc.returncode}. Cek URL/token lalu jalankan lagi.')

    nenen88 = _sm_import_downloader()
    total = len(records)
    for idx, rec in enumerate(records, 1):
        name = rec.get('filename') or Path(urlparse(rec['url']).path).name or rec['url']
        print(f'  [{idx}/{total}] ✓ {name}')
        if rec.get('civitai_json') and rec.get('filename'):
            try:
                nenen88.civitai_infotags(rec['civitai_json'], rec['target'], rec['filename'], rec.get('version_id'))
                threading.Thread(
                    target=nenen88.civitai_preview,
                    args=(rec['civitai_json'], rec['target'], rec['filename'], rec.get('version_id')),
                    daemon=True,
                ).start()
            except Exception as exc:
                print(f"  Metadata Civitai dilewati untuk {name}: {exc}")

def _sm_download_serial(entries):
    cwd = Path.cwd()
    try:
        for line, target in entries:
            target = Path(target)
            target.mkdir(parents=True, exist_ok=True)
            os.chdir(target)
            get_ipython().run_line_magic('download', str(line))
    finally:
        os.chdir(cwd)

def _sm_download_many(entries, parallel=True, max_workers=3):
    clean_entries = [(str(line).strip(), Path(target)) for line, target in entries if str(line).strip()]
    if not clean_entries:
        print('  Tidak ada URL yang diisi.')
        return

    if not parallel or int(max_workers) <= 1:
        _sm_download_serial(clean_entries)
        return

    records, fallback = [], []
    for line, target in clean_entries:
        rec = _sm_resolve_record(line, target)
        if not rec:
            continue
        if rec.get('fallback'):
            fallback.append((line, target))
        else:
            records.append(rec)

    _sm_run_aria(records, max_workers=max_workers)

    if fallback:
        print('  Google Drive atau URL khusus dijalankan dengan downloader standar.')
        _sm_download_serial(fallback)

def _sm_clone_one(line, target_dir):
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    raw = str(line).strip()
    if not raw:
        return 'empty'
    parts = shlex.split(raw)
    if not (len(parts) >= 2 and parts[0] == 'git' and parts[1] == 'clone'):
        parts = ['git', 'clone'] + parts
    url = next((p for p in parts if p.startswith('http://') or p.startswith('https://') or p.startswith('git@')), None)
    repo_name = None
    if url:
        repo_name = Path(urlparse(url).path).name.replace('.git', '')
    if repo_name and (target_dir / repo_name).exists():
        subprocess.run(['git', '-C', str(target_dir / repo_name), 'pull', '--ff-only'], check=False)
        return f'updated {repo_name}'
    subprocess.run(parts, cwd=str(target_dir), check=True)
    return f'cloned {repo_name or raw}'

def _sm_clone_many(lines, target_dir, parallel=True, max_workers=3):
    repos = _sm_strip(lines)
    if not repos:
        print('  Tidak ada extension/custom node yang diisi.')
        return
    target_dir = _sm_path('Extensions', target_dir)
    if not parallel or int(max_workers) <= 1:
        for repo in repos:
            print('  ' + _sm_clone_one(repo, target_dir))
        return
    from concurrent.futures import ThreadPoolExecutor, as_completed
    max_workers = max(1, int(max_workers))
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_sm_clone_one, repo, target_dir): repo for repo in repos}
        for fut in as_completed(futures):
            try:
                print('  ' + fut.result())
            except Exception as exc:
                print(f'  clone gagal: {futures[fut]} -> {exc}')


In [ ]:
# @title <b><font color='orange'>Model Downloader - 5 Checkpoint + 5 LoRA + VAE</font></b> { display-mode: "form" }
# @markdown <details open><summary><b>Checkpoints</b></summary>
Checkpoint_1 = '' # @param { type: "string", placeholder: "URL or leave empty" }
Checkpoint_2 = '' # @param { type: "string", placeholder: "URL or leave empty" }
Checkpoint_3 = '' # @param { type: "string", placeholder: "URL or leave empty" }
Checkpoint_4 = '' # @param { type: "string", placeholder: "URL or leave empty" }
Checkpoint_5 = '' # @param { type: "string", placeholder: "URL or leave empty" }
# @markdown </details>
# @markdown <details open><summary><b>LoRA</b></summary>
Lora_1 = '' # @param { type: "string", placeholder: "URL or leave empty" }
Lora_2 = '' # @param { type: "string", placeholder: "URL or leave empty" }
Lora_3 = '' # @param { type: "string", placeholder: "URL or leave empty" }
Lora_4 = '' # @param { type: "string", placeholder: "URL or leave empty" }
Lora_5 = '' # @param { type: "string", placeholder: "URL or leave empty" }
# @markdown </details>
# @markdown <details open><summary><b>VAE</b></summary>
VAE_URL = '' # @param { type: "string", placeholder: "URL or leave empty" }
# @markdown </details>
# @markdown <details open><summary><b>Speed Options</b></summary>
Parallel_Download = True # @param { type: "boolean" }
Max_Workers = 3 # @param { type: "slider", min: 1, max: 10, step: 1 }
# @markdown </details>

if '_sm_download_many' not in globals():
    raise RuntimeError('Jalankan Cell 1 terlebih dahulu agar helper Segsmaker Fast aktif.')

ckpt_urls = _sm_strip([Checkpoint_1, Checkpoint_2, Checkpoint_3, Checkpoint_4, Checkpoint_5])
lora_urls = _sm_strip([Lora_1, Lora_2, Lora_3, Lora_4, Lora_5])
vae_urls = _sm_strip([VAE_URL])

entries = []
entries += [(url, _sm_path('CKPT')) for url in ckpt_urls]
entries += [(url, _sm_path('LORA')) for url in lora_urls]
entries += [(url, _sm_path('VAE')) for url in vae_urls]

_sm_download_many(entries, parallel=Parallel_Download, max_workers=Max_Workers)


In [ ]:
# @title <b><font color='orange'>Extra Assets - Extensions, Embeddings, Upscalers</font></b> { display-mode: "form" }
# @markdown <details open><summary><b>Extensions / ComfyUI Custom Nodes</b></summary>
Extension_1 = '' # @param { type: "string", placeholder: "git clone URL or leave empty" }
Extension_2 = '' # @param { type: "string", placeholder: "git clone URL or leave empty" }
Extension_3 = '' # @param { type: "string", placeholder: "git clone URL or leave empty" }
Extension_4 = '' # @param { type: "string", placeholder: "git clone URL or leave empty" }
Extension_5 = '' # @param { type: "string", placeholder: "git clone URL or leave empty" }
# @markdown </details>
# @markdown <details open><summary><b>Embeddings</b></summary>
Embedding_1 = '' # @param { type: "string", placeholder: "URL or leave empty" }
Embedding_2 = '' # @param { type: "string", placeholder: "URL or leave empty" }
Embedding_3 = '' # @param { type: "string", placeholder: "URL or leave empty" }
# @markdown </details>
# @markdown <details open><summary><b>Upscalers</b></summary>
Upscaler_1 = '' # @param { type: "string", placeholder: "URL or leave empty" }
Upscaler_2 = '' # @param { type: "string", placeholder: "URL or leave empty" }
Upscaler_3 = '' # @param { type: "string", placeholder: "URL or leave empty" }
# @markdown </details>
# @markdown <details open><summary><b>Speed Options</b></summary>
Assets_Parallel_Download = True # @param { type: "boolean" }
Assets_Max_Workers = 3 # @param { type: "slider", min: 1, max: 10, step: 1 }
# @markdown </details>

if '_sm_download_many' not in globals():
    raise RuntimeError('Jalankan Cell 1 terlebih dahulu agar helper Segsmaker Fast aktif.')

_sm_clone_many(
    [Extension_1, Extension_2, Extension_3, Extension_4, Extension_5],
    _sm_path('Extensions'),
    parallel=Assets_Parallel_Download,
    max_workers=Assets_Max_Workers,
)

asset_entries = []
asset_entries += [(url, _sm_path('Embeddings')) for url in _sm_strip([Embedding_1, Embedding_2, Embedding_3])]
asset_entries += [(url, _sm_path('Upscalers')) for url in _sm_strip([Upscaler_1, Upscaler_2, Upscaler_3])]

_sm_download_many(asset_entries, parallel=Assets_Parallel_Download, max_workers=Assets_Max_Workers)


In [ ]:
# @title <b><font color='orange'>FLUX Model Downloader</font></b> { display-mode: "form" }
# @markdown <details open><summary><b>FLUX Variant</b></summary>
FLUX_Variant = 'FLUX.1-schnell (Fast, 4-step)' # @param ["FLUX.1-schnell (Fast, 4-step)", "FLUX.1-dev (Quality, 20-step)"]
# @markdown </details>
# @markdown <details open><summary><b>Component URLs</b></summary>
FLUX_Unet = '' # @param { type: "string", placeholder: "UNet URL or leave empty" }
FLUX_Clip_L = '' # @param { type: "string", placeholder: "CLIP-L URL or leave empty" }
FLUX_T5XXL = '' # @param { type: "string", placeholder: "T5XXL URL or leave empty" }
FLUX_VAE = '' # @param { type: "string", placeholder: "VAE URL or leave empty" }
# @markdown </details>
# @markdown <details open><summary><b>Speed Options</b></summary>
Parallel_FLUX_Download = True # @param { type: "boolean" }
FLUX_Max_Workers = 2 # @param { type: "slider", min: 1, max: 6, step: 1 }
# @markdown </details>

if '_sm_download_many' not in globals():
    raise RuntimeError('Jalankan Cell 1 terlebih dahulu agar helper Segsmaker Fast aktif.')

def _sm_optional_path(primary, secondary=None):
    value = globals().get(primary) or (globals().get(secondary) if secondary else None)
    if value is None:
        raise RuntimeError(f'Folder {primary} belum tersedia untuk WebUI ini.')
    path = Path(value)
    path.mkdir(parents=True, exist_ok=True)
    return path

flux_entries = []
if str(FLUX_Unet).strip():
    flux_entries.append((FLUX_Unet, _sm_optional_path('UNET', 'CKPT')))
if str(FLUX_Clip_L).strip():
    flux_entries.append((FLUX_Clip_L, _sm_optional_path('CLIP', 'TE')))
if str(FLUX_T5XXL).strip():
    flux_entries.append((FLUX_T5XXL, _sm_optional_path('TE', 'CLIP')))
if str(FLUX_VAE).strip():
    flux_entries.append((FLUX_VAE, _sm_path('VAE')))

print(f'  Variant selected: {FLUX_Variant}')
_sm_download_many(flux_entries, parallel=Parallel_FLUX_Download, max_workers=FLUX_Max_Workers)


In [ ]:
''' Controlnet '''
%run $Controlnet_Widget


In [ ]:
# @title <b><font color='orange'>Launcher WebUI</font></b> { display-mode: "form" }
# @markdown Select the same WebUI that you installed in the first cell.
Software = 'Forge-Neo' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
# @markdown <br><b>Tunnel Tokens</b> — optional.
Ngrok_Token = '' # @param { type: "string", placeholder: "ngrok token or leave empty" }
Zrok_Token = '' # @param { type: "string", placeholder: "zrok token or leave empty" }
# @markdown <br><b>Additional Settings</b>
Extra_Args = '' # @param { type: "string", placeholder: "extra launch args or leave empty" }
Skip_ComfyUI_Check = False # @param { type: "boolean" }
Skip_Widget = False # @param { type: "boolean" }

import json, shlex, os
from pathlib import Path

DEFAULT_ARGS = {
    'A1111': '--xformers',
    'Forge': '--disable-xformers --opt-sdp-attention --cuda-stream',
    'ReForge': '--xformers --cuda-stream',
    'ReForge-old': '--xformers --cuda-stream',
    'Forge-Classic': '--xformers --cuda-stream --persistent-patches',
    'Forge-Neo': '--xformers --cuda-malloc --cuda-stream',
    'ComfyUI': '--dont-print-server --use-pytorch-cross-attention',
    'SwarmUI': '--launch_mode none',
}

try:
    from KANDANG import HOMEPATH
    home = Path(HOMEPATH)
except Exception:
    home = Path('/content')

webui_path = home / Software
if not webui_path.exists():
    raise RuntimeError(f'{Software} belum terinstal di {webui_path}. Jalankan Cell 1 dengan WebUI yang sama.')

marker = home / 'gutris1' / 'marking.json'
if marker.exists():
    data = json.loads(marker.read_text())
    data['ui'] = Software
    marker.write_text(json.dumps(data, indent=4))

if Skip_Widget:
    os.environ['SEGSMAKER_SKIP_WIDGET'] = '1'

launch_args = DEFAULT_ARGS.get(Software, '')
if str(Extra_Args).strip():
    launch_args = f'{launch_args} {Extra_Args.strip()}'.strip()
if str(Ngrok_Token).strip():
    launch_args = f'{launch_args} --N={shlex.quote(Ngrok_Token.strip())}'.strip()
if str(Zrok_Token).strip():
    launch_args = f'{launch_args} --Z={shlex.quote(Zrok_Token.strip())}'.strip()
if Skip_ComfyUI_Check:
    launch_args = f'--skip-comfyui-check {launch_args}'.strip()

%cd -q $webui_path
%run segsmaker.py $launch_args
